---------

In [ ]:
import os
import pandas as pd
import json 
from openai import OpenAI

In [ ]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

------

### Without function 
Example dummy function hard coded to return the same weather  

In [ ]:
messages = [{"role": "user", 
             "content": "What's the weather like in San Francisco, Tokyo, and Paris?"}]

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=messages
    # tools=tools,
    # tool_choice="auto"  # auto is default, but we'll be explicit
)
response.choices[0].message.content

---------

## Function 1

In [24]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Returns hardcoded weather information for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get weather for.",
                },
                "weather": {
                    "type": "string",
                    "description": "The weather in the city.",
                    "enum": ["sunny", "rainy"],
                }
            },
            "required": ["city"]
        }
    }
}

In [25]:
# A prompt that instructs the model to use the extract_student_info function tool.
prompt = f"""I want to know the weather of the city madrid"""

In [26]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=prompt,
    tools=weather_tool,
    # tool_choice="auto"  # auto is default, but we'll be explicit
)
response.choices[0].message.content

BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'messages': expected an array of objects, but got a string instead.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'invalid_type'}}

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
    {
      "role": "user",
      "content": prompt
    }
      ]
)

In [ ]:
response.choices[0].message.content

In [ ]:
json.loads(response.choices[0].message.content)

### Now lets use a function

In [ ]:
student_custom_function = [
    {
        'name': 'extract_student_info',
        'description': 'Get the student information from the body of the input text',
        'parameters': {
            'type': 'object',
            'properties': {
                'name': {
                    'type': 'string',
                    'description': 'Name of the person'
                },
                'college': {
                    'type': 'string',
                    'description': 'The college name.'
                },
                'grades': {
                    'type': 'integer',
                    'description': 'CGPA of the student.'
                },
                'club': {
                    'type': 'string',
                    'description': 'college club for extracurricular activities. '
                }
                
            }
        }
    }
]

In [ ]:
response2 = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt }],
    functions=student_custom_function
)

In [ ]:
response2.choices[0].message

In [ ]:
response2.choices[0].message.function_call

In [ ]:
json.loads(response2.choices[0].message.function_call.arguments)

In [ ]:
student_description_two="krish naik is a student of computer science at IIT Mumbai. He is an indian and has a 9.5 GPA. krish is known for his programming skills and is an active member of the college's data science Club. He hopes to pursue a career in artificial intelligence after graduating."

In [ ]:
student_description_three="sudhanshu kumar is a student of computer science at IIT bengalore. He is an indian and has a 9.2 GPA. krish is known for his programming skills and is an active member of the college's MLops Club. He hopes to pursue a career in artificial intelligence after graduating."

In [ ]:
import json
student_info = [student_description, student_description_two,student_description_three]
for student in student_info:
    response =  client.chat.completions.create(
        model = 'gpt-3.5-turbo',
        messages = [{'role': 'user', 'content': student}],
        functions = student_custom_function,
        function_call = 'auto'
    )

    response = json.loads(response.choices[0].message.function_call.arguments)
    print(response)#import csv

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
    {
      "role": "user",
      "content": "When's the next flight from delhi to mumbai?"
    }
      ]
)
response.choices[0].message.content

In [ ]:
function_descriptions = [
    {
        "name": "get_flight_info",
        "description": "Get flight information between two locations",
        "parameters": {
            "type": "object",
            "properties": {
                "loc_origin": {
                    "type": "string",
                    "description": "The departure airport, e.g. DEL",
                },
                "loc_destination": {
                    "type": "string",
                    "description": "The destination airport, e.g. MUM",
                },
            },
            "required": ["loc_origin", "loc_destination"],
        },
    }
]

In [ ]:
user_prompt = "When's the next flight from new delhi to mumbai?"

In [ ]:
response2 = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
    {
      "role": "user",
      "content": user_prompt
    }
      ],
    # Add function calling
    functions=function_descriptions,
    function_call="auto",  # specify the function call
    
)
response2.choices[0].message.content

In [ ]:
response.choices[0].message

---------

## Function 2 not working

In [ ]:
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

### Step 1: send the conversation and available functions to the model

In [ ]:
tools = [{"type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather in a given location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "unit": {
                            "type": "string", 
                            "enum": ["celsius", "fahrenheit"]},
                    },
                    "required": ["location"],
                },
            },
        }
    ]

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=messages,
    tools=tools,
    tool_choice="auto",  # auto is default, but we'll be explicit
)
print(response.choices[0].message.content)

In [ ]:
response.choices[0]

In [ ]:
response.choices[0].finish_reason

In [ ]:
response.choices[0].message.tool_calls

In [ ]:
response_message = response.choices[0].message
tool_calls = response_message.tool_calls
response_message

### Step 2: check if the model wanted to call a function

In [ ]:
# Messages before
messages = [{"role": "user", "content": "What's the weather like in San Francisco, Tokyo, and Paris?"}]
messages

In [ ]:
# Hacemos un append con la respuesta del LLM diciendo que tiene que utilizar herramientas
messages.append(response_message)  # extend conversation with assistant's reply
messages

In [ ]:
# Step 3: call the function
# Note: the JSON response may not always be valid; be sure to handle errors
available_functions = {
    "get_current_weather": get_current_weather,
}  # only one function in this example, but you can have multiple


---------